# Receipt -> JSON with a QLoRA fine-tuned Vision-Language Model

Turn a photo of a receipt into **schema-validated JSON**.

**What this notebook proves.** It does not just fine-tune a model and claim it
worked. It evaluates the *same held-out receipts* four ways, so the improvement
is measured rather than asserted:

| # | Approach | What it tests |
|---|----------|---------------|
| 1 | OCR + rules (Tesseract) | Do you even need a neural model? |
| 2 | VLM zero-shot | How good is the base model out of the box? |
| 3 | VLM few-shot | Can prompting alone close the gap? |
| 4 | **VLM + QLoRA (ours)** | Does fine-tuning add anything beyond prompting? |

That ladder is the point of the project. Anyone can fine-tune; showing *why*
fine-tuning was the right choice is the harder and more interesting part.

---

### How to run
1. **Runtime -> Change runtime type -> T4 GPU** (free tier is enough)
2. **Runtime -> Run all**

Runs end-to-end in roughly 45-60 minutes. If the CORD dataset fails to
download, the notebook generates synthetic receipts instead so it never
dead-ends during a demo.

*Aditya Raj*

## 0. Setup

Colab starts from a blank VM every session, so dependencies are installed here
rather than assumed.

In [ ]:
# Colab VMs are wiped between sessions, so install every run.
!pip install -q -U "transformers>=4.45.0" "accelerate>=0.34.0" "peft>=0.13.0" \
    "bitsandbytes>=0.44.0" "datasets>=3.0.0" "qwen-vl-utils>=0.0.8" \
    "pydantic>=2.7.0" pytesseract

# Tesseract binary for the OCR baseline (the python package is only a wrapper).
!apt-get -qq install -y tesseract-ocr > /dev/null

print("done")
print()
print("If Colab says the session restarted, that is EXPECTED - these packages")
print("replace versions Colab had already imported. Just Runtime -> Run all")
print("again; this cell is a no-op the second time and nothing is lost.")

In [ ]:
import gc, io, json, os, random, re, sys, textwrap, time
from dataclasses import dataclass

import psutil
import torch
import pandas as pd
from PIL import Image, ImageDraw

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# --- hardware check -------------------------------------------------------
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU.")

GPU_NAME = torch.cuda.get_device_name(0)
TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9

# The free Colab T4 is a Turing card: it has NO bfloat16 support.
# Training in bf16 there silently falls back or errors, so pick the dtype
# from the hardware instead of hard-coding it.
SUPPORTS_BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16

print(f"GPU        : {GPU_NAME}  ({TOTAL_VRAM:.1f} GB)")
print(f"bf16 support: {SUPPORTS_BF16}  ->  training dtype = {DTYPE}")


# --- memory helpers -------------------------------------------------------
# Two separate budgets get confused constantly. Host RAM (~12 GB on Colab) holds
# the dataset; VRAM (~15 GB on a T4) holds the model. Running out of either kills
# the session, but the fixes are completely different, so measure both.

def free_memory():
    gc.collect()
    torch.cuda.empty_cache()


def memory_report(label=""):
    ram = psutil.virtual_memory()
    print(f"[{label:<16}] RAM {ram.used / 1e9:5.1f}/{ram.total / 1e9:.1f} GB"
          f"   VRAM {torch.cuda.memory_allocated() / 1e9:5.1f}/{TOTAL_VRAM:.1f} GB")


memory_report("startup")

In [ ]:
# ---------------------------------------------------------------- config
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"   # 2B fits a 16GB T4 once quantised to 4-bit

# Image tokens, not text, dominate memory: attention cost grows with the SQUARE
# of sequence length. Qwen2-VL turns each 28x28 pixel block into one token, so
# max_pixels directly caps the number of image tokens.
MAX_IMAGE_TOKENS = 256                    # ~= a 448x448 image
MIN_PIXELS = 64 * 28 * 28
MAX_PIXELS = MAX_IMAGE_TOKENS * 28 * 28

# Host-RAM guard. CORD receipts are multi-megapixel photos, so 400 of them held
# as decoded PIL objects is GBs - on a VM with ~12 GB that alone ends the
# session, and it looks like a GPU problem when it is not. The processor
# downsamples everything to MAX_PIXELS anyway, so we shrink once on ingest and
# keep the result JPEG-COMPRESSED, decoding only at the moment of use.
STORE_MAX_SIDE = 1024
STORE_QUALITY = 90

N_TRAIN = 400        # small on purpose: this is a demo run on a free GPU
N_EVAL = 40          # generation is slow, so keep the eval set modest
MAX_STEPS = 200      # bounded so a Colab session cannot time out mid-run
MAX_NEW_TOKENS = 384

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
BATCH_SIZE = 1       # one image per step ...
GRAD_ACCUM = 8       # ... but an effective batch of 8 via accumulation

OUTPUT_DIR = "qwen2vl-receipt-qlora"
print("config set")

## 1. The schema

The model's job is to produce **this exact shape**, every time. Defining it as
a Pydantic model rather than a docstring means malformed output is *detectable*
rather than something we hope does not happen.

This logic lives in `receipt_schema.py` in the repo rather than in the notebook,
because it needs no GPU and is therefore unit-tested in CI (`pytest tests/`).

In [ ]:
# Pull the tested helper module out of the repo. Falls back to a direct
# download if git is unavailable, so the notebook still runs standalone.
REPO = "https://github.com/aditya1609/receipt-to-json-vlm.git"
RAW = "https://raw.githubusercontent.com/aditya1609/receipt-to-json-vlm/main/receipt_schema.py"

if not os.path.exists("receipt_schema.py"):
    if os.system(f"git clone -q {REPO} _repo 2>/dev/null") == 0:
        os.system("cp _repo/receipt_schema.py .")
    else:
        os.system(f"wget -q {RAW} -O receipt_schema.py")

if not os.path.exists("receipt_schema.py"):
    raise SystemExit(
        "Could not fetch receipt_schema.py - upload it manually via the Files panel."
    )

from receipt_schema import (
    INSTRUCTION, SCHEMA_HINT, LineItem, Receipt,
    aggregate, extract_json, normalize_cord, score_prediction,
)

print("Target schema:", SCHEMA_HINT)
print()
print("Prompt sent with every image:")
print(textwrap.indent(INSTRUCTION, "    "))

## 2. Data — CORD receipts

[CORD](https://huggingface.co/datasets/naver-clova-ix/cord-v2) is ~1000
photographed Indonesian receipts, each with structured ground truth already
attached. No annotation work required.

`normalize_cord()` flattens their nested format into our schema so that the
training target and the model's output are directly comparable — you cannot
score a prediction against a differently-shaped gold answer.

### A note on the *other* memory budget

Everyone worries about VRAM when fine-tuning and forgets host RAM, which on a
Colab VM is the smaller of the two. CORD receipts are 2–4 megapixel photos:
decoded, 400 of them is several GB before the model has loaded at all.

Three rules follow, and all three are applied below:

- **Downscale on ingest, not at use time.** The processor caps every image at
  `MAX_PIXELS` (~448×448) regardless, so a full-size original costs tens of MB
  of RAM to deliver 0.2 MP of signal.
- **Store compressed, decode on demand.** A decoded PIL image is an
  *uncompressed* pixel buffer: width × height × 3 bytes. At 1024×768 that is
  ~2.4 MB, against ~150 KB for the same picture as JPEG. Keeping the bytes and
  decoding at the point of use costs a few milliseconds per training step and
  saves an order of magnitude of RAM — the right trade when RAM is the binding
  constraint and the GPU is the bottleneck anyway.
- **Never hold two copies.** Building a list of originals and *then* mapping
  over it to build a second list means both exist simultaneously — the peak is
  what kills you, not the steady state.

The dataset is also *streamed* rather than downloaded, since we need ~440 of
roughly 1000 rows and the local Arrow conversion is itself a RAM spike.

`memory_report()` prints both budgets at each stage so this is observable
rather than guesswork.

In [ ]:
from datasets import load_dataset


def to_jpeg_bytes(image):
    """Downscale, then keep the image COMPRESSED in memory.

    This is the fix for the RAM crash, and it is worth understanding why. A
    decoded 1024x768 RGB image is width x height x 3 = ~2.4 MB of raw bytes,
    because PIL holds an uncompressed pixel buffer. The same picture as JPEG is
    ~150 KB. Across 440 receipts that is ~1 GB versus ~70 MB.

    The cost is decoding at the point of use - a few milliseconds, once per
    training step, which is nothing beside a forward pass through a 2B model.
    Trading a little CPU for an order of magnitude of RAM is the right way round
    on a machine where RAM is the binding constraint.
    """
    image = image.convert("RGB")
    if max(image.size) > STORE_MAX_SIDE:
        image.thumbnail((STORE_MAX_SIDE, STORE_MAX_SIDE), Image.LANCZOS)
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=STORE_QUALITY)
    return buffer.getvalue()


def as_image(example):
    """Decode a stored example back into a PIL image, on demand."""
    return Image.open(io.BytesIO(example["image"])).convert("RGB")


def prepare(rows):
    """Normalise the ground truth and compress, one example at a time.

    Accepts any iterable and keeps only the compressed bytes, so each original
    is freed as soon as the next arrives rather than accumulating.
    """
    out = []
    for row in rows:
        gold = normalize_cord(row["ground_truth"])
        if not gold.items and gold.total is None:
            continue                      # unusable ground truth - drop it
        out.append({"image": to_jpeg_bytes(row["image"]),
                    "gold": gold,
                    "target": gold.to_json()})
    return out


def make_synthetic_receipt(idx):
    """Fallback so a failed download never breaks a live demo."""
    items = random.sample(
        [("NASI GORENG", 40000), ("ES TEH", 8000), ("AYAM BAKAR", 55000),
         ("KOPI SUSU", 18000), ("ROTI BAKAR", 22000), ("JUS ALPUKAT", 25000)],
        k=random.randint(2, 4),
    )
    img = Image.new("RGB", (420, 560), "white")
    draw = ImageDraw.Draw(img)
    draw.text((120, 30), "TOKO DEMO", fill="black")
    draw.text((30, 70), "-" * 45, fill="black")
    y, total = 110, 0
    rows = []
    for name, price in items:
        qty = random.randint(1, 2)
        line_total = price * qty
        total += line_total
        draw.text((30, y), f"{name[:18]:<20}{qty} x{line_total:>9,}", fill="black")
        rows.append({"nm": name, "cnt": f"{qty} x", "price": f"{line_total:,}"})
        y += 34
    draw.text((30, y + 20), "-" * 45, fill="black")
    draw.text((30, y + 55), f"{'TOTAL':<20}{total:>11,}", fill="black")
    gt = {"gt_parse": {"menu": rows,
                       "sub_total": {"subtotal_price": f"{total:,}"},
                       "total": {"total_price": f"{total:,}"}}}
    return {"image": img, "ground_truth": json.dumps(gt)}


def load_cord():
    """Fetch CORD, preferring streaming.

    We need ~440 of roughly 1000 rows. Downloading the full dataset also means
    converting it to Arrow locally, and that conversion is itself a RAM spike -
    so streaming avoids a cost that has nothing to do with the rows we want.
    Falls back to a normal download if streaming is unavailable.
    """
    try:
        stream = load_dataset("naver-clova-ix/cord-v2", streaming=True)
        train = prepare(stream["train"].take(N_TRAIN))
        held_out = prepare(stream["test"].take(N_EVAL))
        if train and held_out:
            return train, held_out, "CORD (streamed)"
        raise RuntimeError("stream yielded no usable rows")
    except Exception as exc:
        print(f"   streaming unavailable ({type(exc).__name__}) - downloading instead")

    ds = load_dataset("naver-clova-ix/cord-v2")
    train = prepare(ds["train"].select(range(min(N_TRAIN, len(ds["train"])))))
    held_out = prepare(ds["test"].select(range(min(N_EVAL, len(ds["test"])))))
    del ds
    return train, held_out, "CORD"


memory_report("before data")

try:
    train_data, eval_data, SOURCE = load_cord()
except Exception as exc:
    print(f"CORD unavailable ({type(exc).__name__}) - using synthetic receipts.")
    train_data = prepare(make_synthetic_receipt(i) for i in range(N_TRAIN))
    eval_data = prepare(make_synthetic_receipt(10_000 + i) for i in range(N_EVAL))
    SOURCE = "synthetic"

USING_SYNTHETIC = SOURCE == "synthetic"
free_memory()
print(f"usable ({SOURCE}): {len(train_data)} train / {len(eval_data)} eval")
memory_report("after data")

In [ ]:
example = train_data[0]
image = as_image(example)

# Show the saving explicitly - this is the number that decides whether the
# session survives, so it is worth printing rather than assuming.
n_images = len(train_data) + len(eval_data)
held_mb = sum(len(ex["image"]) for ex in train_data + eval_data) / 1e6
decoded_mb = held_mb * (image.width * image.height * 3) / max(len(example["image"]), 1)
print(f"stored size   : {image.size}")
print(f"held in RAM   : {held_mb:.0f} MB compressed, across {n_images} images")
print(f"would have been: ~{decoded_mb:.0f} MB if kept decoded")

print("\nGold JSON the model must learn to produce:")
print(json.dumps(json.loads(example["target"]), indent=2)[:600])
display(image.resize((300, int(300 * image.height / image.width))))

## 3. Baseline 1 — OCR + rules

**The question every interviewer asks: "why not just use OCR?"**

Answering it with numbers instead of opinion is worth the twenty minutes this
takes. Tesseract reads text perfectly well; what it cannot do is tell you
*which* number is the total, which is a line item, and which is a phone number.
That is a layout-and-semantics problem, not a character-recognition one.

In [ ]:
import pytesseract

TOTAL_WORDS = ("total", "jumlah", "amount", "grand")
SKIP_WORDS = ("subtotal", "sub total", "sub-total", "cash", "change", "kembali",
              "tunai", "npwp", "tax", "ppn", "discount", "diskon")
PRICE_RE = re.compile(r"(\d[\d.,]{2,})\s*$")

def ocr_extract(image):
    """Best-effort rule-based extraction: the honest non-neural baseline."""
    try:
        text = pytesseract.image_to_string(image)
    except Exception:
        return "{}"

    items, total, subtotal = [], None, None
    for line in (l.strip() for l in text.splitlines()):
        if not line:
            continue
        low = line.lower()
        match = PRICE_RE.search(line)
        if not match:
            continue
        from receipt_schema import parse_money
        value = parse_money(match.group(1))
        if value is None:
            continue
        label = line[: match.start()].strip(" .:-")

        if any(w in low for w in ("subtotal", "sub total", "sub-total")):
            subtotal = value
        elif any(w in low for w in TOTAL_WORDS):
            total = value
        elif label and not any(w in low for w in SKIP_WORDS):
            items.append({"name": label, "quantity": None, "price": value})

    return json.dumps({"items": items, "subtotal": subtotal, "total": total})


def evaluate(predict_fn, data, label, show=0):
    """Run a prediction function over the eval set and average the metrics."""
    scores, rows = [], []
    start = time.time()
    for i, ex in enumerate(data):
        raw = predict_fn(ex)
        score = score_prediction(raw, ex["gold"])
        scores.append(score)
        rows.append({"raw": raw, **score})
        if i < show:
            print(f"--- {label} sample {i} ---")
            print(raw[:300])
            print()
        if (i + 1) % 10 == 0:
            # Reclaim the fragmented cache; also proof of life, since generating
            # 40 receipts takes minutes and a silent cell looks like a hang.
            free_memory()
            print(f"   {label}: {i + 1}/{len(data)}"
                  f"   ({(time.time() - start) / (i + 1):.1f}s each)", flush=True)
    summary = aggregate(scores)
    summary["seconds_per_receipt"] = (time.time() - start) / max(len(data), 1)
    return summary, rows


results = {}
results["1. OCR + rules"], ocr_rows = evaluate(
    lambda ex: ocr_extract(as_image(ex)), eval_data, "OCR", show=1
)
pd.DataFrame([results["1. OCR + rules"]]).round(3)

## 4. Load the VLM in 4-bit

**QLoRA in one sentence:** freeze the base model and store it in 4-bit so it
barely uses memory, then train a small set of new weights on top in higher
precision.

A 2B model in fp16 needs ~4.4 GB just for weights, before activations and
optimiser state. In 4-bit it needs closer to 1.5 GB, which is what leaves room
on a T4 for the image tokens and gradients.

In [ ]:
from transformers import (AutoProcessor, BitsAndBytesConfig,
                          Qwen2VLForConditionalGeneration)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NF4: a datatype shaped for normally-distributed weights
    bnb_4bit_compute_dtype=DTYPE,       # maths still happens in fp16/bf16
    bnb_4bit_use_double_quant=True,     # quantise the quantisation constants too
)

free_memory()
memory_report("before model")

processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=DTYPE,
    # Pin every layer to the one GPU. "auto" is allowed to offload layers to
    # host RAM when it thinks VRAM is tight, which trades a clean OOM for a
    # silently 10x slower run - and eats the RAM the dataset needs.
    device_map={"": 0},
    low_cpu_mem_usage=True,   # stream shards straight to the GPU, no full CPU copy
)
model.config.use_cache = False           # required alongside gradient checkpointing

print(f"loaded {MODEL_ID}")
memory_report("after model")

In [ ]:
IMAGE_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")

def build_messages(image, few_shot=None):
    """Chat-format one request. few_shot = list of (image, target_json) pairs."""
    messages = []
    for shot_img, shot_target in (few_shot or []):
        messages.append({"role": "user", "content": [
            {"type": "image", "image": shot_img},
            {"type": "text", "text": INSTRUCTION}]})
        messages.append({"role": "assistant", "content": [
            {"type": "text", "text": shot_target}]})
    messages.append({"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": INSTRUCTION}]})
    return messages


@torch.inference_mode()
def generate(image, few_shot=None):
    """Greedy decode - deterministic, which matters when comparing runs."""
    messages = build_messages(image, few_shot)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images = [s[0] for s in (few_shot or [])] + [image]

    inputs = processor(text=[text], images=images, return_tensors="pt").to(model.device)
    generated = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        # Overrides config.use_cache, which is off for training. Without the KV
        # cache every new token re-attends over the whole prompt - hundreds of
        # image tokens - making generation quadratic and far slower.
        use_cache=True,
    )
    # Strip the prompt: keep only the newly generated tokens.
    trimmed = generated[0][inputs["input_ids"].shape[1]:]
    decoded = processor.decode(trimmed, skip_special_tokens=True)
    del inputs, generated, trimmed   # drop the KV cache now, not at next collection
    return decoded


# sanity check on one image
print(generate(as_image(eval_data[0]))[:400])

## 5. Baseline 2 — zero-shot

The base model with no examples and no training. Expect valid-ish JSON some of
the time, wrapped in conversational text, with inconsistent field names.

**Measuring this before training is non-negotiable.** It is the only way to
prove later that the fine-tune did anything.

In [ ]:
results["2. VLM zero-shot"], zero_rows = evaluate(
    lambda ex: generate(as_image(ex)), eval_data, "zero-shot", show=2
)
pd.DataFrame([results["2. VLM zero-shot"]]).round(3)

## 6. Baseline 3 — few-shot

Two worked examples in the prompt, still no weight updates. This usually fixes
*formatting* — the model copies the shape it just saw — while doing much less
for *reading accuracy*.

If few-shot were enough, fine-tuning would be wasted effort. This cell is what
lets you say "I checked" instead of assuming.

In [ ]:
# Only two images, decoded once and reused for every query - worth holding.
FEW_SHOT = [(as_image(ex), ex["target"]) for ex in train_data[:2]]

results["3. VLM few-shot"], few_rows = evaluate(
    lambda ex: generate(as_image(ex), few_shot=FEW_SHOT), eval_data, "few-shot", show=1
)
pd.DataFrame([results["3. VLM few-shot"]]).round(3)

## 7. QLoRA fine-tuning

Two decisions worth defending in an interview:

**The vision encoder stays frozen.** It already sees receipts perfectly well —
what the model lacks is knowledge of *our output format*. Adapting the language
side is cheaper and targets the actual problem. Note how the LoRA target names
(`q_proj`, `gate_proj`, ...) only exist in the language model; Qwen2-VL's vision
tower uses different names (`qkv`, `fc1`), so it is excluded automatically.

**Loss is computed only on the answer.** The prompt is identical on every
example, so training the model to predict it would waste capacity on something
already known. Prompt tokens get label `-100`, which PyTorch ignores.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    # These names appear only in the language model, never the vision tower.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Verify the vision tower really was left alone.
adapted = {n.split(".lora_")[0] for n, _ in model.named_parameters() if "lora_" in n}
vision_hits = [m for m in adapted if "visual" in m]
print(f"\nmodules with adapters: {len(adapted)}")
print(f"vision-tower modules adapted: {len(vision_hits)}  (expected 0)")

In [ ]:
def collate_fn(examples):
    """Build a training batch and mask everything we do not want loss on."""
    texts, images, prompt_lens = [], [], []

    for ex in examples:
        image = as_image(ex)          # decoded here, discarded when the batch is done
        messages = build_messages(image)
        prompt = processor.apply_chat_template(messages, tokenize=False,
                                               add_generation_prompt=True)
        texts.append(prompt + ex["target"] + "<|im_end|>\n")
        images.append(image)
        # Length of the prompt *including* its expanded image tokens, so we know
        # exactly where the answer starts in the final sequence.
        prompt_ids = processor(text=[prompt], images=[image],
                               return_tensors="pt")["input_ids"]
        prompt_lens.append(prompt_ids.shape[1])

    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100   # ignore padding
    labels[labels == IMAGE_TOKEN_ID] = -100                     # ignore image placeholders
    for i, length in enumerate(prompt_lens):
        labels[i, :length] = -100                               # ignore the prompt
    batch["labels"] = labels
    return batch


# Confirm the masking is right: only the JSON answer should survive.
_check = collate_fn([train_data[0]])
_kept = _check["labels"][0][_check["labels"][0] != -100]
print("tokens contributing to loss:", len(_kept))
print("decoded ->", processor.decode(_kept)[:200])

In [ ]:
from transformers import Trainer, TrainingArguments

# Plain Trainer + a custom collator, rather than TRL's SFTTrainer: multimodal
# batches need bespoke collation anyway, and this has a far more stable API.
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="no",
    optim="paged_adamw_8bit",        # 8-bit optimiser states: big VRAM saving
    gradient_checkpointing=True,
    fp16=(DTYPE == torch.float16),
    bf16=(DTYPE == torch.bfloat16),
    remove_unused_columns=False,     # our "columns" are JPEG bytes, not tensors
    report_to="none",                # set to "wandb" for the LoRA rank sweep
    seed=SEED,
    # Worker processes fork the parent, and copy-on-write does not survive
    # Python refcounting - each worker would end up with its own copy of the
    # in-memory image list. Load in-process instead.
    dataloader_num_workers=0,
    dataloader_pin_memory=False,     # pinned buffers are locked host RAM we cannot spare
)

trainer = Trainer(model=model, args=args,
                  train_dataset=train_data, data_collator=collate_fn)

free_memory()
memory_report("before train")
train_result = trainer.train()
print(f"\nfinal training loss: {train_result.training_loss:.4f}")

# The optimiser states are ~2 adapter-sized tensors per parameter and are dead
# weight during evaluation, so drop the trainer before generating again.
del trainer, train_result
free_memory()
memory_report("after train")

## 8. Evaluate the fine-tuned model

Identical eval set, identical prompt, identical decoding. The only thing that
changed is the adapter weights — which is what makes the comparison honest.

In [ ]:
model.eval()
model.config.use_cache = True
free_memory()

results["4. VLM + QLoRA"], tuned_rows = evaluate(
    lambda ex: generate(as_image(ex)), eval_data, "fine-tuned", show=2
)

comparison = pd.DataFrame(results).T[
    ["json_valid", "schema_valid", "total_correct", "subtotal_correct",
     "item_name_f1", "item_price_accuracy", "seconds_per_receipt"]
].round(3)
print("\n=== THE ADAPTATION LADDER ===")
comparison

In [ ]:
# The headline chart for the README.
ax = comparison[["json_valid", "total_correct", "item_name_f1"]].plot(
    kind="bar", figsize=(9, 4.5), rot=12,
    title="Receipt extraction: OCR vs prompting vs fine-tuning",
)
ax.set_ylabel("score (0-1)")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right")
ax.figure.tight_layout()
ax.figure.savefig("results_comparison.png", dpi=150)
print("saved results_comparison.png - put this in the README")

## 9. Error analysis

A single accuracy number hides everything useful. Breaking results down per
field turns "87% accurate" into "96% on totals but 71% on line items, because
X" — which is the difference between reporting a number and understanding a
model.

In [ ]:
tuned = pd.DataFrame(tuned_rows)

print("Per-field accuracy (fine-tuned):")
for field in ["json_valid", "schema_valid", "total_correct",
              "subtotal_correct", "item_count_correct"]:
    print(f"   {field:22s} {tuned[field].mean():.1%}")
print(f"   {'item_name_f1':22s} {tuned['item_name_f1'].mean():.1%}")

failures = tuned[~tuned["total_correct"]]
print(f"\n{len(failures)} of {len(tuned)} receipts got the TOTAL wrong.")

for idx in failures.index[:3]:
    print(f"\n--- failure {idx} ---")
    print("gold :", eval_data[idx]["target"][:200])
    print("pred :", str(failures.loc[idx, "raw"])[:200])

In [ ]:
# Does accuracy depend on how much is on the receipt? A visible downward trend
# means long receipts are the weak point - likely a token-budget problem.
tuned["n_gold_items"] = [len(ex["gold"].items) for ex in eval_data]
by_size = tuned.groupby("n_gold_items")[["item_name_f1", "total_correct"]].agg(["mean", "count"])
print("Accuracy by receipt length:")
by_size.round(3)

## 10. Validation and retry

The model is a *probabilistic* system feeding a *deterministic* one. Anything
downstream — a database, an accounting system — needs guarantees the model
cannot give.

So the output is parsed and validated against the Pydantic schema, and a
failure triggers one retry at a higher temperature before giving up honestly.
Never let unvalidated model output reach a database.

The stronger version of this is *constrained decoding* (libraries like
Outlines), which makes invalid JSON structurally impossible rather than
something you catch afterwards. Retry-on-failure is the simpler cousin and a
good thing to be able to compare against.

In [ ]:
@dataclass
class Extraction:
    receipt: "Receipt | None"
    attempts: int
    ok: bool
    error: str = ""


def extract_receipt(image, max_attempts=2) -> Extraction:
    """Generate, validate, retry once, then fail loudly rather than silently."""
    last_error = ""
    for attempt in range(1, max_attempts + 1):
        raw = generate(image)
        payload = extract_json(raw)
        if payload is None:
            last_error = "no JSON object found in output"
            continue
        try:
            return Extraction(Receipt.model_validate(payload), attempt, True)
        except Exception as exc:
            last_error = f"schema validation failed: {type(exc).__name__}"
    return Extraction(None, max_attempts, False, last_error)


ok = 0
for ex in eval_data[:10]:
    result = extract_receipt(as_image(ex))
    ok += result.ok
print(f"{ok}/10 receipts produced schema-valid output within 2 attempts")

demo = extract_receipt(as_image(eval_data[0]))
print("\nvalidated object:", demo.receipt)

## 11. Try your own receipt

Upload any receipt photo. This is the cell to run live in an interview.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        img = Image.open(name).convert("RGB")
        display(img.resize((320, int(320 * img.height / img.width))))
        result = extract_receipt(img)
        if result.ok:
            print(json.dumps(result.receipt.model_dump(), indent=2))
        else:
            print("failed validation:", result.error)
except ImportError:
    print("Not running in Colab - skip this cell.")

## 12. Publish the adapter

The adapter is only a few megabytes because LoRA trains ~1% of the parameters —
that is precisely why it can be shared when the full model cannot. Weights are
gitignored; the Hub is the right home for them.

In [ ]:
# Needs a token from huggingface.co/settings/tokens (write scope).
PUSH_TO_HUB = False          # flip to True when you are ready
HUB_REPO = "aditya1609/qwen2vl-2b-receipt-json-lora"

model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
size_mb = sum(
    os.path.getsize(os.path.join(OUTPUT_DIR, f)) for f in os.listdir(OUTPUT_DIR)
) / 1e6
print(f"adapter saved locally: {size_mb:.1f} MB (vs ~4400 MB for the full model)")

if PUSH_TO_HUB:
    from huggingface_hub import login
    login()
    model.push_to_hub(HUB_REPO)
    processor.push_to_hub(HUB_REPO)
    print("pushed ->", HUB_REPO)

## 13. What to say about this in an interview

**The pitch (60 seconds).**
> I fine-tuned a 2-billion-parameter vision-language model to turn receipt
> photos into schema-validated JSON, using QLoRA on a single free GPU. The
> model is a ViT feeding a language model through a learned projector, so the
> image arrives as tokens the LLM can attend over — I train LoRA adapters on
> the language side and leave the vision encoder frozen, which is about 1% of
> parameters. What I care about most is that I measured four approaches on the
> same held-out set: an OCR baseline, zero-shot, few-shot, and the fine-tune.
> That tells me fine-tuning was actually worth it rather than assumed.

**Questions this project invites, and where the answer lives.**

| Question | Section |
|---|---|
| Why not just use OCR? | 3 — with numbers |
| Why fine-tune instead of prompting? | 5, 6, 8 — the ladder |
| Why LoRA and not full fine-tuning? | 7 — 1% of params, one 16 GB GPU, no forgetting |
| Why freeze the vision encoder? | 7 — it already sees; formatting is the gap |
| What is QLoRA doing exactly? | 4 — NF4 4-bit frozen base, adapters in fp16 |
| How did you fit this on a T4? | 4, 7 — capped image tokens, checkpointing, 8-bit optimiser |
| Why mask the prompt in the labels? | 7 — loss belongs on the answer only |
| How do you know it works? | 8, 9 — per-field metrics, not one number |
| What breaks it? | 9 — error analysis by field and receipt length |
| What if it emits bad JSON? | 10 — validate, retry, fail loudly |

**Be honest about the limits.** ~400 training examples and 200 steps is a demo
run. It is one language and one receipt style. There is no human-review queue,
and receipts contain personal data, so a real deployment needs a redaction and
retention policy. Knowing what is missing is worth as much as what is built.